# 🏏 CricketIQ 2.0 — Notebook 05: Batter vs Bowler (BvB) Matchup Engine
## BCA Machine Learning Project Pipeline — Phase 5

**Objective:**
1. Formulate the **Net BvB Impact Index** that jointly balances scoring rate against dismissal threat.
2. Compile granular head-to-head records across 240,000+ historical balls.
3. Compute matchup dominance ratings (Batter Dominance vs Bowler Dominance vs Balanced).


### Net BvB Impact Index Formula

$$\text{Net BvB Impact Index} = \frac{\text{Runs} - (18 \times \text{Dismissals})}{\text{Balls}} \times 100$$

* **Par Benchmark:** $\approx +40.0$ index points.
* **Score $> +85.0$:** 🟢 Batter Dominance.
* **Score $< +15.0$:** 🔴 Bowler Dominance.


In [1]:
import sys
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve() if Path("..").resolve().name == "CricketIQ 2.0" else Path(".").resolve()
sys.path.insert(0, str(BASE_DIR))

from src.data_loader import load_data
from src.player_features import build_player_stats, get_bvb_stats

matches, deliveries = load_data(BASE_DIR)
_, bvb_history = build_player_stats(matches, deliveries)

pairs = [
    ('V Kohli', 'JJ Bumrah'),
    ('V Kohli', 'Mohammed Shami'),
    ('RG Sharma', 'TA Boult'),
    ('AD Russell', 'Rashid Khan'),
    ('MS Dhoni', 'B Kumar'),
    ('VR Iyer', 'JJ Bumrah')
]

h2h_list = []
for bat, bowl in pairs:
    h = get_bvb_stats(bvb_history, bat, bowl)
    dom = "🟢 Batter Edge" if h['bvb_net_impact'] > 60 else ("🔴 Bowler Edge" if h['bvb_net_impact'] < 25 else "⚖️ Balanced")
    h2h_list.append({
        'Batter': bat,
        'Bowler': bowl,
        'Balls': h['bvb_balls'],
        'Runs': h['bvb_runs'],
        'Dismissals': h['bvb_dismissals'],
        'Shrunk SR': round(h['bvb_sr'], 1),
        'Shrunk Avg': round(h['bvb_avg'], 1),
        'Net Impact Index': round(h['bvb_net_impact'], 1),
        'Dominance': dom
    })

display(pd.DataFrame(h2h_list))


,Batter,Bowler,Balls,Runs,Dismissals,Shrunk SR,Shrunk Avg,Net Impact Index,Dominance
0,V Kohli,JJ Bumrah,98,145,5,144.1,26.4,56.1,⚖️ Balanced
1,V Kohli,Mohammed Shami,80,107,5,132.0,21.6,21.2,🔴 Bowler Edge
2,RG Sharma,TA Boult,25,23,3,106.7,14.8,-124.0,🔴 Bowler Edge
3,AD Russell,Rashid Khan,23,24,3,114.0,15.0,-130.4,🔴 Bowler Edge
4,MS Dhoni,B Kumar,61,94,0,146.9,53.3,154.1,🟢 Batter Edge
5,VR Iyer,JJ Bumrah,28,33,2,120.8,19.8,-10.7,🔴 Bowler Edge


### Intelligent Frontline Bowler Selection Algorithm

In head-to-head cricket modeling, selecting candidate bowlers by raw or uncalibrated wicket-rate causes part-timers (e.g. Glenn Phillips with 12 career balls and 2 wickets) or pure batters (e.g. Phil Salt with 0 balls bowled) to wrongfully appear as primary bowlers while excluding elite pacers like **Mohammed Siraj** (93 IPL wickets, 1,800+ balls).

We design **`select_bvb_bowlers()`** to enforce:
1. **Disqualification of Pure Batters:** Any player without bowling specialization who has bowled fewer than 60 balls (10 overs) is disqualified.
2. **Specialist Priority & Workload Ranking:** Prioritizes specialist bowlers (`role == 'bowl'`) and ranks by proven career bowling volume (`bowl_balls + bowl_wickets * 10`).
3. **Pace & Spin Composition:** Produces the team's top 5 authentic frontline wicket-takers (e.g., 3 pacers + 2 spinners).


In [2]:
from src.squad_data import get_squad_data
from src.player_features import build_global_player_stats

global_stats = build_global_player_stats(matches, deliveries)
squad_data = get_squad_data(BASE_DIR)

def role_of(player: str, gs: dict) -> str:
    s = gs.get(player, {})
    ib = s.get("is_batter", False)
    iw = s.get("is_bowler", False)
    if ib and iw:  return "all"
    if iw:         return "bowl"
    if ib:         return "bat"
    return "bat"

def select_bvb_bowlers(xi: list[str], gs: dict, n: int = 5) -> list[str]:
    candidates = []
    for p in xi:
        s = gs.get(p, {})
        role = role_of(p, gs)
        balls = s.get("bowl_balls", 0)
        wkts = s.get("bowl_wickets", 0)
        
        # Disqualify non-bowlers / pure batters
        if role == "bat" and balls < 60:
            continue
        if balls == 0 and role != "bowl":
            continue
            
        score = (3000 if role == "bowl" else (1500 if role == "all" else 0)) + balls + (wkts * 10)
        candidates.append((p, score, balls, wkts, role))
        
    candidates.sort(key=lambda x: x[1], reverse=True)
    bowlers = [p for p, _, _, _, _ in candidates[:n]]
    
    if len(bowlers) < n:
        for p in reversed(xi):
            if p not in bowlers:
                bowlers.append(p)
            if len(bowlers) == n:
                break
    return bowlers[:n]

def select_bvb_batters(xi: list[str], gs: dict, n: int = 5) -> list[str]:
    batters = []
    for p in xi:
        s = gs.get(p, {})
        role = role_of(p, gs)
        if role in ("bat", "all") or s.get("bat_runs", 0) >= 100:
            batters.append(p)
        if len(batters) == n:
            break
    if len(batters) < n:
        for p in xi:
            if p not in batters:
                batters.append(p)
            if len(batters) == n:
                break
    return batters[:n]

# Test on Gujarat Titans & Royal Challengers Bengaluru
gt_xi = squad_data["Gujarat Titans"]["last_xi"]
rcb_xi = squad_data["Royal Challengers Bengaluru"]["last_xi"]

gt_bowlers = select_bvb_bowlers(gt_xi, global_stats, 5)
gt_batters = select_bvb_batters(gt_xi, global_stats, 5)
rcb_bowlers = select_bvb_bowlers(rcb_xi, global_stats, 5)
rcb_batters = select_bvb_batters(rcb_xi, global_stats, 5)

print("🏆 Verified Top 5 Frontline Bowlers for BvB:")
print(f" • Gujarat Titans Bowlers: {', '.join(gt_bowlers)}")
print(f" • Royal Challengers Bengaluru Bowlers: {', '.join(rcb_bowlers)}")
print("\n🏏 Verified Top 5 Primary Batsmen for BvB:")
print(f" • Gujarat Titans Batters: {', '.join(gt_batters)}")
print(f" • Royal Challengers Bengaluru Batters: {', '.join(rcb_batters)}")

# Verification checks
assert 'Mohammed Siraj' in gt_bowlers, "Mohammed Siraj must be in GT bowlers!"
assert 'GD Phillips' not in gt_bowlers, "Glenn Phillips must NOT be in bowlers!"
assert 'PD Salt' not in rcb_bowlers, "Phil Salt must NOT be in RCB bowlers!"
print("\n✅ All bowler authority & batter qualification assertions passed!")


🏆 Verified Top 5 Frontline Bowlers for BvB:
 • Gujarat Titans Bowlers: Rashid Khan, K Rabada, Mohammed Siraj, R Sai Kishore, Washington Sundar
 • Royal Challengers Bengaluru Bowlers: B Kumar, KH Pandya, Yash Dayal, JR Hazlewood, LS Livingstone

🏏 Verified Top 5 Primary Batsmen for BvB:
 • Gujarat Titans Batters: Shubman Gill, JC Buttler, B Sai Sudharsan, M Shahrukh Khan, GD Phillips
 • Royal Challengers Bengaluru Batters: V Kohli, PD Salt, RM Patidar, LS Livingstone, JM Sharma

✅ All bowler authority & batter qualification assertions passed!
